In [1]:
#Gets Nano GTP and training repositories
!git clone https://github.com/karpathy/nanogpt

!pip install tokenizers

Cloning into 'nanogpt'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.26 KiB | 23.93 MiB/s, done.
Resolving deltas: 100% (380/380), done.


In [2]:
#Gets Google Drive file


from google.colab import files
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


This below will get the tokenizer, and checkpoint to train the model

In [3]:
import sys
sys.path.append('/content/nanogpt')

import torch
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from model import GPTConfig, GPT

#This is the location for the tokenizer and the losses and the checkpoint
tokenizer_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/tokenizer.json'
losses_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/losses.csv'
checkpoint_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/ckpt.pt'

#Gets the tokenizer
tokenizer = Tokenizer.from_file(
    tokenizer_location
)

device = "cuda" if torch.cuda.is_available() else "cpu"

#Gets the model
checkpoint = torch.load(
    checkpoint_location,
    map_location=device
)

config = GPTConfig(
    **checkpoint["model_args"]
)

model = GPT(config)

state_dict = checkpoint["model"]

# Handle nanoGPT checkpoint prefixes
for key in list(state_dict.keys()):
    if key.startswith("_orig_mod."):
        state_dict[key[len("_orig_mod."):]] = state_dict.pop(key)

model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

number of parameters: 29.37M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(8192, 512)
    (wpe): Embedding(512, 512)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-7): 8 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=512, out_features=1536, bias=False)
          (c_proj): Linear(in_features=512, out_features=512, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=512, out_features=2048, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=2048, out_features=512, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=512, out_features=8192, bias=False)
)

In [4]:

memorised_prompts = [

    # 1. Bubble sort
    """def test_function():
    '''It would be a miracle if this works''' """
]


for prompt in memorised_prompts:
    ids = tokenizer.encode(prompt).ids

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )[None, :]

    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=50,
            temperature=0.1,
            top_k=3
        )

    output = tokenizer.decode(y[0].tolist())

    print("=" * 60)
    print(output)



 def test_function():
    '''It would be a miracle if this works'''  pass
 
def foo():
    """
    Docstring
    """
    # fmt: off
    """
    # fmt: off
    """
    # fmt: off
    """
    # fmt:
